# Feature Engineering

This notebook creates engineered features from the preprocessed telco churn dataset to improve model performance and business insights.

In [1]:
# Imports
import pandas as pd
import numpy as np
from pathlib import Path
import pickle

In [2]:
# Configuration
DATA_DIR = Path("../data/processed")
OUTPUT_DIR = Path("../data/processed")

# Load preprocessed data
X_train = pd.read_parquet(DATA_DIR / "train.parquet")
X_test = pd.read_parquet(DATA_DIR / "test.parquet")
y_train = pd.read_parquet(DATA_DIR / "train_labels.parquet").iloc[:, 0]
y_test = pd.read_parquet(DATA_DIR / "test_labels.parquet").iloc[:, 0]

print(f"Training data shape: {X_train.shape}")
print(f"Test data shape: {X_test.shape}")

Training data shape: (5625, 46)
Test data shape: (1407, 46)


In [3]:
# Load original data for feature engineering (before one-hot encoding)
original_data = pd.read_csv("../data/raw/telco_customer_churn.csv")
original_data['TotalCharges'] = pd.to_numeric(original_data['TotalCharges'], errors='coerce')
original_data = original_data.dropna()

# Match the same train/test split
from sklearn.model_selection import train_test_split
train_ids = X_train['customerID']
test_ids = X_test['customerID']

original_train = original_data[original_data['customerID'].isin(train_ids)].copy()
original_test = original_data[original_data['customerID'].isin(test_ids)].copy()

print(f"Original train shape: {original_train.shape}")
print(f"Original test shape: {original_test.shape}")

Original train shape: (5625, 21)
Original test shape: (1407, 21)


## 1. Derived Features

In [4]:
def create_derived_features(df):
    """Create derived features from original data."""
    df = df.copy()
    
    # Average monthly charges (avoid division by zero)
    df['avg_monthly_charges'] = np.where(
        df['tenure'] > 0, 
        df['TotalCharges'] / df['tenure'], 
        df['MonthlyCharges']
    )
    
    # Service count (number of active services)
    service_cols = ['PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 
                   'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
    
    # Convert to binary (Yes=1, No/No service=0)
    for col in service_cols:
        df[col] = (df[col] == 'Yes').astype(int)
    
    df['services_count'] = df[service_cols].sum(axis=1)
    
    # Has internet service
    df['has_internet'] = (df['InternetService'] != 'No').astype(int)
    
    # Contract type numeric (Month-to-month=0, One year=1, Two year=2)
    contract_map = {'Month-to-month': 0, 'One year': 1, 'Two year': 2}
    df['contract_numeric'] = df['Contract'].map(contract_map)
    
    # Payment method stability (automatic=1, manual=0)
    df['payment_automatic'] = df['PaymentMethod'].str.contains('automatic').astype(int)
    
    # Is senior citizen with partner
    df['senior_with_partner'] = ((df['SeniorCitizen'] == 1) & (df['Partner'] == 'Yes')).astype(int)
    
    return df

# Apply to train and test
train_derived = create_derived_features(original_train)
test_derived = create_derived_features(original_test)

print(f"Derived features train shape: {train_derived.shape}")
print(f"Derived features test shape: {test_derived.shape}")

Derived features train shape: (5625, 27)
Derived features test shape: (1407, 27)


## 2. Interaction Features

In [5]:
def create_interaction_features(df):
    """Create interaction features."""
    df = df.copy()
    
    # Tenure x Monthly charges (customer lifetime value proxy)
    df['tenure_x_monthly'] = df['tenure'] * df['MonthlyCharges']
    
    # Tenure x Contract type (loyalty score)
    df['tenure_x_contract'] = df['tenure'] * df['contract_numeric']
    
    # Internet service x Monthly charges
    df['internet_x_monthly'] = df['has_internet'] * df['MonthlyCharges']
    
    # Services count x Monthly charges
    df['services_x_monthly'] = df['services_count'] * df['MonthlyCharges']
    
    # Senior citizen x Contract type
    df['senior_x_contract'] = df['SeniorCitizen'] * df['contract_numeric']
    
    # Dependents x Tenure (family stability)
    df['dependents_x_tenure'] = ((df['Dependents'] == 'Yes').astype(int)) * df['tenure']
    
    return df

# Apply to train and test
train_interactions = create_interaction_features(train_derived)
test_interactions = create_interaction_features(test_derived)

print(f"With interactions train shape: {train_interactions.shape}")
print(f"With interactions test shape: {test_interactions.shape}")

With interactions train shape: (5625, 33)
With interactions test shape: (1407, 33)


## 3. Binned Features

In [6]:
def create_binned_features(df):
    """Create binned/categorical features."""
    df = df.copy()
    
    # Tenure groups
    df['tenure_group'] = pd.cut(
        df['tenure'], 
        bins=[0, 12, 24, 48, 72], 
        labels=['0-12', '13-24', '25-48', '49-72'],
        include_lowest=True
    )
    
    # Monthly charges groups
    df['charges_group'] = pd.cut(
        df['MonthlyCharges'], 
        bins=[0, 35, 70, 100, 200], 
        labels=['Low', 'Medium', 'High', 'Very High'],
        include_lowest=True
    )
    
    # Total charges groups
    df['total_charges_group'] = pd.cut(
        df['TotalCharges'], 
        bins=[0, 500, 2000, 5000, 10000], 
        labels=['Low', 'Medium', 'High', 'Very High'],
        include_lowest=True
    )
    
    # Age groups (based on SeniorCitizen)
    df['age_group'] = np.where(df['SeniorCitizen'] == 1, 'Senior', 'Adult')
    
    return df

# Apply to train and test
train_binned = create_binned_features(train_interactions)
test_binned = create_binned_features(test_interactions)

print(f"With binned features train shape: {train_binned.shape}")
print(f"With binned features test shape: {test_binned.shape}")

With binned features train shape: (5625, 37)
With binned features test shape: (1407, 37)


## 4. Business Logic Features

In [7]:
def create_business_features(df):
    """Create business logic features."""
    df = df.copy()
    
    # Customer lifetime value proxy
    df['clv_proxy'] = df['tenure'] * df['MonthlyCharges']
    
    # Risk score based on service combinations
    # High risk: Month-to-month + Electronic check + Fiber optic
    df['high_risk_combo'] = (
        (df['Contract'] == 'Month-to-month') & 
        (df['PaymentMethod'] == 'Electronic check') & 
        (df['InternetService'] == 'Fiber optic')
    ).astype(int)
    
    # Loyalty score (tenure + contract type)
    df['loyalty_score'] = df['tenure'] + (df['contract_numeric'] * 24)
    
    # Service complexity score
    df['complexity_score'] = df['services_count'] + df['has_internet']
    
    # Premium customer indicator
    df['is_premium'] = (
        (df['MonthlyCharges'] > df['MonthlyCharges'].median()) & 
        (df['contract_numeric'] >= 1)
    ).astype(int)
    
    # New customer indicator
    df['is_new_customer'] = (df['tenure'] <= 12).astype(int)
    
    return df

# Apply to train and test
train_final = create_business_features(train_binned)
test_final = create_business_features(test_binned)

print(f"Final engineered features train shape: {train_final.shape}")
print(f"Final engineered features test shape: {test_final.shape}")

Final engineered features train shape: (5625, 43)
Final engineered features test shape: (1407, 43)


## 5. Feature Selection and Final Processing

In [8]:
# Select engineered features (exclude original columns that will cause conflicts)
engineered_cols = [
    'avg_monthly_charges', 'services_count', 'has_internet', 'contract_numeric',
    'payment_automatic', 'senior_with_partner', 'tenure_x_monthly', 'tenure_x_contract',
    'internet_x_monthly', 'services_x_monthly', 'senior_x_contract', 'dependents_x_tenure',
    'tenure_group', 'charges_group', 'total_charges_group', 'age_group',
    'clv_proxy', 'high_risk_combo', 'loyalty_score', 'complexity_score',
    'is_premium', 'is_new_customer'
]

train_engineered = train_final[['customerID'] + engineered_cols].copy()
test_engineered = test_final[['customerID'] + engineered_cols].copy()

print(f"Engineered features only train shape: {train_engineered.shape}")
print(f"Engineered features only test shape: {test_engineered.shape}")
print(f"\nEngineered features: {len(engineered_cols)}")
print(f"Features: {engineered_cols}")

Engineered features only train shape: (5625, 23)
Engineered features only test shape: (1407, 23)

Engineered features: 22
Features: ['avg_monthly_charges', 'services_count', 'has_internet', 'contract_numeric', 'payment_automatic', 'senior_with_partner', 'tenure_x_monthly', 'tenure_x_contract', 'internet_x_monthly', 'services_x_monthly', 'senior_x_contract', 'dependents_x_tenure', 'tenure_group', 'charges_group', 'total_charges_group', 'age_group', 'clv_proxy', 'high_risk_combo', 'loyalty_score', 'complexity_score', 'is_premium', 'is_new_customer']


In [9]:
# Encode categorical engineered features
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

# Identify categorical vs numerical engineered features
categorical_engineered = ['tenure_group', 'charges_group', 'total_charges_group', 'age_group']
numerical_engineered = [col for col in engineered_cols if col not in categorical_engineered]

print(f"Categorical engineered features: {len(categorical_engineered)}")
print(f"Numerical engineered features: {len(numerical_engineered)}")

# Encode categorical features
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Fit on training data
train_cat = train_engineered[categorical_engineered]
encoder.fit(train_cat)

# Transform both train and test
train_encoded = encoder.transform(train_cat)
test_encoded = encoder.transform(test_engineered[categorical_engineered])

# Create DataFrames
train_encoded_df = pd.DataFrame(
    train_encoded,
    index=train_engineered.index,
    columns=encoder.get_feature_names_out()
)

test_encoded_df = pd.DataFrame(
    test_encoded,
    index=test_engineered.index,
    columns=encoder.get_feature_names_out()
)

print(f"Encoded categorical train shape: {train_encoded_df.shape}")
print(f"Encoded categorical test shape: {test_encoded_df.shape}")

Categorical engineered features: 4
Numerical engineered features: 18
Encoded categorical train shape: (5625, 14)
Encoded categorical test shape: (1407, 14)


In [10]:
# Scale numerical engineered features
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit on training data
train_num = train_engineered[numerical_engineered]
scaler.fit(train_num)

# Transform both train and test
train_scaled = scaler.transform(train_num)
test_scaled = scaler.transform(test_engineered[numerical_engineered])

# Create DataFrames
train_scaled_df = pd.DataFrame(
    train_scaled,
    index=train_engineered.index,
    columns=numerical_engineered
)

test_scaled_df = pd.DataFrame(
    test_scaled,
    index=test_engineered.index,
    columns=numerical_engineered
)

print(f"Scaled numerical train shape: {train_scaled_df.shape}")
print(f"Scaled numerical test shape: {test_scaled_df.shape}")

Scaled numerical train shape: (5625, 18)
Scaled numerical test shape: (1407, 18)


In [11]:
# Combine engineered features with original processed features
# Load original processed data (without customerID for merging)
X_train_original = X_train.drop(columns=['customerID'])
X_test_original = X_test.drop(columns=['customerID'])

# Combine all features
train_final_features = pd.concat([
    X_train_original.reset_index(drop=True),
    train_scaled_df.reset_index(drop=True),
    train_encoded_df.reset_index(drop=True)
], axis=1)

test_final_features = pd.concat([
    X_test_original.reset_index(drop=True),
    test_scaled_df.reset_index(drop=True),
    test_encoded_df.reset_index(drop=True)
], axis=1)

# Add customerID back
train_final_features.insert(0, 'customerID', X_train['customerID'].values)
test_final_features.insert(0, 'customerID', X_test['customerID'].values)

print(f"Final train features shape: {train_final_features.shape}")
print(f"Final test features shape: {test_final_features.shape}")
print(f"\nTotal features: {train_final_features.shape[1] - 1}")  # -1 for customerID

Final train features shape: (5625, 78)
Final test features shape: (1407, 78)

Total features: 77


## 6. Save Engineered Features

In [12]:
# Save final engineered datasets
train_engineered_path = OUTPUT_DIR / "train_engineered.parquet"
test_engineered_path = OUTPUT_DIR / "test_engineered.parquet"

train_final_features.to_parquet(train_engineered_path)
test_final_features.to_parquet(test_engineered_path)

print(f"Saved engineered training data: {train_engineered_path}")
print(f"Saved engineered test data: {test_engineered_path}")

Saved engineered training data: ../data/processed/train_engineered.parquet
Saved engineered test data: ../data/processed/test_engineered.parquet


In [13]:
# Save feature engineering objects
engineered_objects_path = OUTPUT_DIR / "feature_engineering_objects.pkl"

engineering_objects = {
    'encoder': encoder,
    'scaler': scaler,
    'engineered_features': engineered_cols,
    'categorical_engineered': categorical_engineered,
    'numerical_engineered': numerical_engineered
}

with open(engineered_objects_path, 'wb') as f:
    pickle.dump(engineering_objects, f)

print(f"Saved feature engineering objects: {engineered_objects_path}")

Saved feature engineering objects: ../data/processed/feature_engineering_objects.pkl


## 7. Feature Engineering Summary

In [14]:
print("=== Feature Engineering Summary ===")
print(f"\nOriginal features: {X_train_original.shape[1]}")
print(f"Engineered features: {len(engineered_cols)}")
print(f"Total features after engineering: {train_final_features.shape[1] - 1}")
print(f"\nFeature types:")
print(f"- Derived features: 7")
print(f"- Interaction features: 6")
print(f"- Binned features: 4")
print(f"- Business logic features: 6")
print(f"\nCategorical engineered: {len(categorical_engineered)}")
print(f"Numerical engineered: {len(numerical_engineered)}")

print(f"\nSample of engineered features:")
print(train_engineered[engineered_cols[:5]].head())

=== Feature Engineering Summary ===

Original features: 45
Engineered features: 22
Total features after engineering: 77

Feature types:
- Derived features: 7
- Interaction features: 6
- Binned features: 4
- Business logic features: 6

Categorical engineered: 4
Numerical engineered: 18

Sample of engineered features:
   avg_monthly_charges  services_count  has_internet  contract_numeric  \
0            29.850000               1             1                 0   
1            55.573529               3             1                 1   
2            54.075000               3             1                 0   
3            40.905556               3             1                 1   
4            75.825000               1             1                 0   

   payment_automatic  
0                  0  
1                  0  
2                  0  
3                  1  
4                  0  
